# TabNet tự giám sát

Đo lợi ích của tiền huấn luyện trên TabNet, một kiến trúc bảng chọn đặc trưng theo nhiều bước.

So sánh cấu hình `tabnet-masked` với `tabnet-scratch` trên RTX 4060.

## 1. Chọn chế độ

- `view` đọc kết quả đã lưu. `VIEW_RESULTS` chọn thư mục `full` hoặc `smoke`.
- `smoke` kiểm tra luồng chạy với dữ liệu rút gọn, 2 fold, 1 epoch tiền huấn luyện và 1 epoch phân loại. Đối chứng chỉ chạy 1 epoch phân loại.
- `full` dùng 699.960 dòng train, 299.984 dòng test và 5 fold. Trình chạy tiếp tục từ các fold có kết quả hoàn chỉnh với cùng cấu hình.

Khi chạy huấn luyện, chọn kernel của môi trường PyTorch CUDA đã chuẩn bị cho NVIDIA RTX 4060.

In [1]:
from pathlib import Path

METHOD = 'tabnet-masked'
CONTROL = 'tabnet-scratch'
RUN_MODE = "view"  # "view", "smoke" hoặc "full"
VIEW_RESULTS = "full"  # Đổi thành "smoke" để xem một lượt chạy thử

if RUN_MODE not in {"view", "smoke", "full"}:
    raise ValueError("RUN_MODE phải là view, smoke hoặc full.")
if VIEW_RESULTS not in {"smoke", "full"}:
    raise ValueError("VIEW_RESULTS phải là smoke hoặc full.")

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "s6e11_gpu" / "pretraining").is_dir():
        project_root = candidate
        break
else:
    raise RuntimeError("Mở notebook trong thư mục dự án hoặc một thư mục con.")

experiment_dir = project_root / "s6e11_gpu" / "pretraining"
runner = experiment_dir / "run_pretraining.py"
result_root = experiment_dir / "results" / (
    VIEW_RESULTS if RUN_MODE == "view" else RUN_MODE
)
print(f"Phương pháp: {METHOD} | Đối chứng: {CONTROL} | Kết quả: {result_root.name}")

Phương pháp: tabnet-masked | Đối chứng: tabnet-scratch | Kết quả: full


## 2. Dữ liệu và phạm vi của fold

- Dùng 24 đặc trưng của S6E11, gồm 13 cột số, 5 cột phân loại và 6 cờ. `id` dùng để ghép kết quả, còn `is_fraud` là nhãn phân loại.
- Lượt đầy đủ giữ cùng 5 fold phân tầng với mô hình đối chứng trong dự án. Encoder được học riêng từ phần train của từng fold. Các dòng validation và test được biến đổi bằng cấu hình đã học từ phần train.
- Giai đoạn tiền huấn luyện học từ các cột đầu vào. Giai đoạn tinh chỉnh học từ nhãn `is_fraud` của phần train. Validation ngoài fold dùng để chấm điểm sau khi huấn luyện.

## 3. Tiền huấn luyện

TabNet bản gọn chọn và tổng hợp đặc trưng qua 3 bước quyết định. Lượt chạy sử dụng thư viện `pytorch-tabnet==4.1.0`.

1. Che từng cột độc lập với xác suất **20%** theo phép lấy mẫu Bernoulli. Số cột bị che thay đổi theo giao dịch.
2. Khôi phục giá trị số và vector của cột phân loại bị che. Sai số bình phương được chia cho phương sai của đặc trưng với mức sàn **0,1**, nhằm giới hạn ảnh hưởng của các cột gần như hằng.
3. Dùng encoder đã học để huấn luyện đầu phân loại gian lận.

[TabNet, AAAI 2021](https://research.google/pubs/tabnet-attentive-interpretable-tabular-learning/) · [Mã mô hình](../tabnet_model.py)

## 4. Tinh chỉnh và đối chứng

Lượt đầy đủ cố định **3 epoch tiền huấn luyện** và **4 epoch phân loại**. Batch size là **4.096** trong bước tiền huấn luyện và **4.096** trong bước phân loại.

Đầu phân loại dự đoán một điểm gian lận cho mỗi dòng. Toàn bộ mạng được cập nhật bằng AdamW với tốc độ học 0,001, tối ưu binary cross-entropy (BCE) trên nhãn `is_fraud`. Đối chứng `tabnet-scratch` khởi tạo ngẫu nhiên với cùng kiến trúc và chạy đúng 4 epoch phân loại trên cùng phép chia fold.

## 5. Chạy trên GPU

Chế độ huấn luyện dùng NVIDIA RTX 4060 qua `cuda:0`. Cell dưới khởi chạy hai cấu hình và in tiến độ từng fold. Chế độ `view` chỉ đọc kết quả ở cell tiếp theo.

In [2]:
if RUN_MODE == "view":
    print("Chế độ xem kết quả đã lưu.")
else:
    import os
    import subprocess
    import sys
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("Hãy chọn kernel của môi trường PyTorch có CUDA.")
    print(f"Thiết bị: {torch.cuda.get_device_name(0)}", flush=True)
    command = [
        sys.executable, "-u", str(runner),
        "--variants", CONTROL, METHOD,
    ]
    if RUN_MODE == "smoke":
        command.append("--smoke")
    environment = dict(os.environ, PYTHONIOENCODING="utf-8")
    with subprocess.Popen(
        command, cwd=project_root, env=environment, text=True,
        encoding="utf-8", stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

Chế độ xem kết quả đã lưu.


## 6. Đọc kết quả đã lưu

**OOF ROC AUC** đo khả năng xếp hạng trên toàn bộ dự đoán ngoài fold. Mỗi dòng được chấm bởi một mô hình học từ các phần khác. Bảng dưới cũng hiển thị **Average Precision**, **Log-loss** và chênh lệch AUC giữa hai cấu hình trên từng fold khi có đủ dữ liệu.

In [3]:
import json
import pandas as pd
from IPython.display import display

records = {}
for variant in (CONTROL, METHOD):
    metric_path = result_root / variant / "metrics.json"
    if metric_path.is_file():
        with metric_path.open(encoding="utf-8") as file:
            records[variant] = json.load(file)
    else:
        print(f"Chưa có kết quả hoàn chỉnh của {variant} trong {result_root.name}.")

if records:
    rows = [
        {"Cấu hình": variant, "OOF ROC AUC": record["oof_auc"],
         "Average Precision": record.get("average_precision"),
         "Log-loss": record.get("log_loss")}
        for variant, record in records.items()
    ]
    display(pd.DataFrame(rows).set_index("Cấu hình").round(6))

    fold_tables = {}
    for variant, record in records.items():
        fold_rows = pd.DataFrame(record.get("folds", []))
        if {"fold", "auc"} <= set(fold_rows.columns):
            fold_tables[variant] = fold_rows.set_index("fold")["auc"]
    if fold_tables:
        by_fold = pd.DataFrame(fold_tables).sort_index()
        if {CONTROL, METHOD} <= set(by_fold.columns):
            by_fold["Δ tiền huấn luyện"] = by_fold[METHOD] - by_fold[CONTROL]
        display(by_fold.round(6))

    if {CONTROL, METHOD} <= records.keys():
        delta = records[METHOD]["oof_auc"] - records[CONTROL]["oof_auc"]
        print(f"Δ OOF ROC AUC so với {CONTROL}: {delta:+.6f}")

    timing_rows = []
    for variant, record in records.items():
        for fold in record.get("folds", []):
            timing_rows.append({
                "Cấu hình": variant, "Fold": fold.get("fold"),
                "Tiền huấn luyện (s)": fold.get("pretrain_seconds"),
                "Tinh chỉnh (s)": fold.get("finetune_seconds"),
                "Thiết bị": fold.get("device"),
            })
    if timing_rows:
        display(pd.DataFrame(timing_rows).round(1))

Δ OOF ROC AUC so với tabnet-scratch: +0.019240


,OOF ROC AUC,Average Precision,Log-loss
Cấu hình,,,
tabnet-scratch,0.733782,0.085424,0.077921
tabnet-masked,0.753023,0.094014,0.077087


,tabnet-scratch,tabnet-masked,Δ tiền huấn luyện
fold,,,
1,0.713220,0.720845,0.007626
2,0.702353,0.757241,0.054889
3,0.763325,0.783168,0.019843
4,0.753229,0.749279,-0.003950
5,0.744434,0.759029,0.014595


,Cấu hình,Fold,Tiền huấn luyện (s),Tinh chỉnh (s),Thiết bị
0,tabnet-scratch,1,0.0,12.9,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
1,tabnet-scratch,2,0.0,13.1,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
2,tabnet-scratch,3,0.0,13.8,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
3,tabnet-scratch,4,0.0,14.4,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
4,tabnet-scratch,5,0.0,13.5,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
5,tabnet-masked,1,7.6,13.1,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
6,tabnet-masked,2,7.7,13.3,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
7,tabnet-masked,3,7.6,13.0,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
8,tabnet-masked,4,7.6,12.8,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
9,tabnet-masked,5,8.2,12.8,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU


## 7. Diễn giải và kết hợp

So với TabNet khởi tạo ngẫu nhiên cùng kiến trúc, sau đó so thời gian và kết quả với MLP và FT-Transformer.

Nếu thử ghép đặc trưng, chỉ dùng encoder tự giám sát đã đóng băng của chính fold đó để bổ sung vector cho `XGB-D03`. Nếu gộp điểm dự đoán, đối chiếu phương án cố định 95% `FINAL` + 5% mô hình mới với `FINAL` trên cùng các fold.

[Tổng hợp kết quả các hướng](../results/summary.md)